# 07 — Frozen-Champion Noise Robustness

Export one success-rate-versus-noise PNG and its aggregate CSV from existing terminal trials. No convergence, ECDF, cross-function plots, synthesis or evaluation runs.


In [1]:
from pathlib import Path
import sys
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "shared").is_dir()), None)
if root is None:
    raise RuntimeError("Open this notebook from the project directory or notebooks/analysis/.")
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

# Notebook-owned presentation helpers are imported from the project root.
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import plotly.io as pio
pio.defaults.mathjax = None
from benchmark.infra.di import BenchmarkProvider
from benchmark.application.benchmark_config import BenchmarkConfig, ReliabilityConfig
from benchmark.infra.storage.analysis_results_store import AnalysisResultsStore
from shared.infra.lifecycle import WorkflowScope
from benchmark.application.analysis.analyze_noise_robustness import AnalyzeNoiseRobustness
from shared.config import RESULTS_DIR
from notebooks.analysis.plotting.exporter import AnalysisFigureExporter
from notebooks.analysis.result_exports import AnalysisResultsExporter


figure_exporter = AnalysisFigureExporter(RESULTS_DIR)

# Optional condition filters; None selects the full available matrix.
FILTER_DIMS = None
FILTER_PROBLEMS = None
FILTER_NOISE_STDS = None


analysis_config = BenchmarkConfig(
    budget=100_000, trials_per_condition=20, random_seed=42, first_instance_id=2,
    trial_timeout_seconds=30.0, observation_trace_points=64,
    classical_baselines=["cmaes", "de", "pso"],
    noise_robustness_enabled=True, cross_function_enabled=False,
    cross_function_problem_ids=[1, 8, 11, 15, 21],
    reliability=ReliabilityConfig(
        primary_prompt_strategy="baseline", primary_target=1e-8, secondary_target=1e-2,
        bootstrap_samples=1000, bootstrap_seed=20260923,
        checkpoint_fractions=[0.01, 0.1, 0.5, 1.0],
    ),
)

# Prepare services once; infrastructure owns shutdown cleanup.
if "services" in globals():
    services.close()
services = WorkflowScope(BenchmarkProvider(), context={}, request_context={BenchmarkConfig: analysis_config})
noise_analysis = services.get(AnalyzeNoiseRobustness)
results_store = services.get(AnalysisResultsStore)
results_exporter = AnalysisResultsExporter(results_store)


Diagnostic log: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/logs/benchmark/2026-10-30_14-45-20_UTC-dcecdab3/diagnostics.log


In [2]:
noise_result = noise_analysis.execute(dims=FILTER_DIMS, problems=FILTER_PROBLEMS, noise_stds=FILTER_NOISE_STDS)
if noise_result.is_success:
    noise = noise_result.value
    noise_snapshot = results_exporter.save(noise)
    display(noise.aggregate)
    print("; ".join(noise.diagnostics) if noise.diagnostics else "Noise statistics calculated; no images exported yet.")
    
    print(f"Saved numerical results: {noise_snapshot}")
else:
    noise = None
    print(f"Error: {noise_result.error.message}")


,Model,Dim,Noise Std,Conditions,Trials,Primary Success Rate,Bootstrap CI Lower,Bootstrap CI Upper
0,gwdg.qwen3-coder-next,2,0.00,5,100,0.79,0.39975,1.00000
1,pso,2,0.00,5,100,0.69,0.40000,0.91000
2,cmaes,2,0.00,5,100,0.62,0.24000,1.00000
3,qwen2.5-coder-32b-instruct-q4_k_m.gguf,2,0.00,5,100,0.59,0.19000,0.99000
4,qwen2.5-coder-14b-instruct-q4_k_m.gguf,2,0.00,5,100,0.21,0.00000,0.61000
5,de,2,0.00,5,100,0.02,0.00000,0.06000
6,cmaes,2,0.05,5,100,0.60,0.30000,0.90000
7,pso,2,0.05,5,100,0.60,0.30000,0.86000
8,de,2,0.05,5,100,0.01,0.00000,0.03000
9,pso,2,0.10,5,100,0.60,0.29000,0.84000


Noise statistics calculated; no images exported yet.
Saved numerical results: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/analysis/noise_robustness/790a5667ff0a6ceb0cccb5c8b8f3f517bb3246b7655b840b451680a05d9d4fe6


## Main plot: success rate versus noise level

Terminal clean returned-point success at the primary target. Baseline strategy only; dimensions are separate panels. Complete functions receive equal weight with 95% condition-bootstrap intervals. Coverage can differ across levels; inspect the exported counts before interpreting a change. Missing conditions are not failures. No plot is exported when only clean trials exist.


In [3]:
if noise is not None:
    figure_exporter.export_noise_robustness(noise)
else:
    print("Complete the preceding workflow successfully before using its data.")


ExportSummary(written=(PosixPath('/Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/figures/05_noise_robustness/success_rate_vs_noise.png'),), skipped=())